# 04 · Rotas com menos buracos

**Rota Livre** · extensão do componente de IA: a IA recomenda o **caminho com menos buracos** entre dois pontos de São Paulo.

> ⚠️ **Dados simulados.** As vias existem, mas as contagens de buracos (`base/vias_simuladas.json`) foram geradas para o MVP.
> Em produção, viriam das solicitações de tapa-buraco do SP156 (Portal de Dados Abertos) e dos relatos do app.

**Índice de buracos** = (buracos abertos + graves) / km — graves contam em dobro; quanto menor, melhor.

Mesmas 3 configurações do projeto: **sem contexto** (LLM sozinha), **RAG** (busca os textos das rotas e a LLM compara os números)
e **agente** (identifica origem/destino, usa a **ferramenta de ranking** com cálculo exato e a LLM só explica, com validação e recusa).

In [ ]:
import sys, json
sys.path.append("..")
import pandas as pd
from src import rotas, llm
print("LLM local disponível?", llm.disponivel())
print(len(rotas.VIAS), "vias ·", len(rotas.ROTAS), "rotas ·", len(rotas.LOCAIS), "locais")

## A base: rotas candidatas por trajeto (ordenadas pelo índice)

In [ ]:
linhas = []
for r in sorted(rotas.ROTAS, key=lambda r: (r["origem"], r["destino"], r["indice"])):
    linhas.append({"trajeto": f"{rotas.nome_local(r['origem'])} → {rotas.nome_local(r['destino'])}", "rota": r["id"],
                   "vias": " → ".join(rotas.VIAS[v]["nome"] for v in r["vias"]),
                   "km": r["km"], "buracos": r["buracos"], "graves": r["graves"], "índice": r["indice"]})
pd.set_option("display.max_colwidth", 80)
pd.DataFrame(linhas)

Repare que **a rota com menos buracos nem sempre é a mais curta** — é exatamente a decisão que o cidadão não consegue tomar sozinho.

## Exemplo: a mesma pergunta nas 3 configurações

In [ ]:
pergunta = "Qual o caminho com menos buracos da Praça da Sé até Pinheiros?"
for modo in ["sem_contexto", "rag", "agente"]:
    r = rotas.responder(pergunta, modo)
    print(f"\n===== {modo} =====")
    print(json.dumps(r["json"], ensure_ascii=False, indent=2))
    for p in r["passos"]:
        print("  ·", p)

## Trajeto fora da base (esperado: "não consta")

In [ ]:
pergunta = "Qual o caminho com menos buracos de Guarulhos até Santo André?"
for modo in ["sem_contexto", "rag", "agente"]:
    r = rotas.responder(pergunta, modo)
    print(f"{modo:>12}:", json.dumps(r["json"], ensure_ascii=False))

## Placar das rotas (12 casos: 8 trajetos da base + 4 fora da base)

In [ ]:
resultados = rotas.avaliar()   # salva avaliacao/rotas_resultados.json e avaliacao/rotas_placar.md
pd.DataFrame([r["metricas"] for r in resultados.values()])

## Erros de cada configuração

In [ ]:
for nome, r in resultados.items():
    erros = [e for e in r["execucoes"] if not e["correto"]]
    print(f"\n===== {nome}: {len(erros)} erros =====")
    for e in erros:
        print(f"- {e['caso_id']} ({e['tipo']}): {json.dumps(e['json'], ensure_ascii=False)[:150]}")

### Interpretação

- **Sem contexto:** sem os dados, o modelo não tem como saber quantos buracos há em cada via — ou recusa tudo, ou inventa ruas e rotas.
- **RAG:** recebe os números das rotas, mas precisa **comparar índices** sozinho; um modelo de 3B erra comparações numéricas e às vezes escolhe a rota mais curta em vez da com menos buracos.
- **Agente:** o cálculo é feito pela **ferramenta** (exato) e a LLM só explica; a validação garante que a rota citada é a do topo do ranking, e trajetos fora da base são recusados.

**Conclusão:** para decisões com números, a especialização por agente com ferramenta é a configuração certa para o MVP.